 **Helena — Agentic AI Assistant**

Helena is a lightweight AI research assistant built using Google's Agent Development Kit (ADK) and Gemini API.
This project demonstrates:
- Tool-augmented AI agents
- LLM orchestration
- Google Search integration
- Runtime execution using ADK
- Real-time information retrieval
The system uses Gemini as its reasoning engine and dynamically invokes Google Search when external information is required.

# Step 1 — Installing Google ADK

In this step, we install Google's Agent Development Kit (ADK).

ADK is a framework for building AI agents that can:
- reason using LLMs
- use external tools
- manage workflows
- execute actions

We use ADK to create a tool-augmented AI assistant powered by Gemini.

In [ ]:
pip install google-adk


# Step 2 — Configuring Gemini API Access

To use Gemini models, authentication is required through a Google API key.

Instead of hardcoding credentials directly into the notebook, Kaggle Secrets are used for secure API key management.

The API key is stored as an environment variable so the ADK framework and Gemini model can access it securely.

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini API key setup complete.")
except Exception as e:
    print(f"Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")
    

Gemini API key setup complete.


# Step 3 — Importing ADK Components

In this step, we import the core components required to build the AI agent.

Key components include:

- `Agent`
    Defines the intelligent agent behavior.

- `Gemini`
    Connects the agent to Google's Gemini LLM.

- `InMemoryRunner`
    Executes the agent workflow and manages runtime state.

- `google_search`
    External search tool enabling real-time information retrieval.

This architecture allows the system to move beyond a traditional chatbot into a tool-augmented AI agent.

In [ ]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("ADK components imported successfully.")

# Step 4 — Configuring API Retry Logic

Large-scale APIs may occasionally fail due to:
- rate limits
- temporary server overload
- network interruptions

To improve robustness, retry logic is added using exponential backoff.

This enables the system to automatically retry failed API requests for specific HTTP error codes such as:
- 429 (Too Many Requests)
- 500 (Internal Server Error)
- 503 (Service Unavailable)
- 504 (Gateway Timeout)

This is a common production engineering practice for reliable AI systems.

In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

# Step 5 — Creating Helper Functions

Kaggle notebooks operate inside a managed cloud environment and do not directly expose local web applications.

The helper functions below generate a proxied URL that allows the ADK Web UI to be accessed through Kaggle infrastructure.

This enables interactive visualization and debugging of the AI agent runtime.

In [ ]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong> IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("Helper functions defined.")

# Step 6 — Building the Helena AI Agent

In this step, the core AI agent is created.

Architecture Components:
- Gemini 2.5 Flash Lite → reasoning engine
- Agent → orchestration layer
- Google Search Tool → external information retrieval

The agent follows a tool-augmented workflow:
1. Receive user query
2. Analyze whether external information is needed
3. Optionally invoke Google Search
4. Generate final response using Gemini

This transforms the system from a static chatbot into an agentic AI assistant capable of real-time information retrieval.

In [ ]:
root_agent = Agent(
    name="Helen",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print(f"Root Agent {root_agent.name}  defined.")
runner = InMemoryRunner(agent=root_agent)

print("Runner created.")

# Step 7 — Initializing the Agent Runner

The `InMemoryRunner` acts as the runtime execution engine for the AI agent.

Responsibilities include:
- managing conversation flow
- coordinating tool execution
- handling agent interactions
- maintaining temporary runtime state

The runner serves as the operational environment where the Helena agent executes user requests.

In [ ]:
response = await runner.run_debug("what is an ai agent?")

# Concepts Explored

This project helped explore several foundational concepts in modern AI engineering:

- Agentic AI
- Tool-Augmented LLMs
- API Integration
- Runtime Orchestration
- Prompt Engineering
- Search-Augmented Generation
- Async Execution
- AI Workflow Design
- Retry and Error Handling
- Secure Credential Management

# Future Improvements

Potential future enhancements for Helena include:

- Long-term memory support
- Multi-agent collaboration
- Voice interaction
- Streamlit-based UI
- PDF and document summarization
- Source citation system
- Autonomous task planning
- Retrieval-Augmented Generation (RAG)